# Non-Instruction Model Evaluation Notebook

This notebook helps you run and understand 5 KPIs for your non-instruction fine-tuned model.

KPIs covered:
1. Perplexity Improvement %
2. Domain Relevance Rate %
3. Hallucination/Off-topic Rate %
4. Repetition Rate %
5. Draft Usability Rate %

## Step 1: Imports and Logging
Run this first.

In [8]:
import logging
from pathlib import Path

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("non_instruction_evaluation_notebook")

print("Imports loaded successfully")

Imports loaded successfully


## Step 2: KPI Definitions
These explanations are shown so you understand each KPI before running calculations.

- perplexity_improvement_percent: How much perplexity improved after fine-tuning compared with baseline. Higher is better.
- domain_relevance_rate_percent: Percent of outputs that stay in finance/credit-card context. Higher is better.
- hallucination_offtopic_rate_percent: Percent of outputs containing fabricated or off-topic content. Lower is better.
- repetition_rate_percent: Percent of outputs with looping/repeated phrases. Lower is better.
- draft_usability_rate_percent: Percent of outputs usable as a first draft with minimal edits. Higher is better.

In [9]:
KPI_DEFINITIONS = {
    "perplexity_improvement_percent": "How much perplexity improved after fine-tuning compared with baseline. Higher is better.",
    "domain_relevance_rate_percent": "Percent of outputs that stay in finance/credit-card context. Higher is better.",
    "hallucination_offtopic_rate_percent": "Percent of outputs containing fabricated or off-topic content. Lower is better.",
    "repetition_rate_percent": "Percent of outputs with looping/repeated phrases. Lower is better.",
    "draft_usability_rate_percent": "Percent of outputs usable as a first draft with minimal edits. Higher is better.",
}

for name, text in KPI_DEFINITIONS.items():
    print(f"- {name}: {text}")

- perplexity_improvement_percent: How much perplexity improved after fine-tuning compared with baseline. Higher is better.
- domain_relevance_rate_percent: Percent of outputs that stay in finance/credit-card context. Higher is better.
- hallucination_offtopic_rate_percent: Percent of outputs containing fabricated or off-topic content. Lower is better.
- repetition_rate_percent: Percent of outputs with looping/repeated phrases. Lower is better.
- draft_usability_rate_percent: Percent of outputs usable as a first draft with minimal edits. Higher is better.


## Step 3: Data Helpers
Use demo data first, then switch to your labeled CSV.

In [ ]:
def build_demo_dataframe() -> pd.DataFrame:
    demo_rows = [
        {
            "generated_text": "Customer reported recurring annual fee dispute and requested charge breakdown.",
            "is_domain_relevant": 1,
            "is_hallucinated_or_offtopic": 0,
            "is_repetitive": 0,
            "is_draft_usable": 1,
        },
        {
            "generated_text": "Unauthorized transaction reported; advise immediate card lock and written dispute.",
            "is_domain_relevant": 1,
            "is_hallucinated_or_offtopic": 0,
            "is_repetitive": 0,
            "is_draft_usable": 1,
        },
        {
            "generated_text": "Unrelated multiple-choice style content appeared in output.",
            "is_domain_relevant": 0,
            "is_hallucinated_or_offtopic": 1,
            "is_repetitive": 0,
            "is_draft_usable": 0,
        },
        {
            "generated_text": "Fee dispute fee dispute fee dispute repeated phrase.",
            "is_domain_relevant": 1,
            "is_hallucinated_or_offtopic": 0,
            "is_repetitive": 1,
            "is_draft_usable": 0,
        },
    ]
    return pd.DataFrame(demo_rows)


def load_user_csv(csv_path: str) -> pd.DataFrame:
    path = Path(csv_path)
    if not path.exists():
        raise FileNotFoundError(f"CSV not found: {path}")
    logger.info("Loading CSV: %s", path)
    return pd.read_csv(path)

## Step 4: Validation and KPI Computation Functions

In [14]:
REQUIRED_COLUMNS = [
    "is_domain_relevant",
    "is_hallucinated_or_offtopic",
    "is_repetitive",
    "is_draft_usable",
]


def validate_required_columns(dataframe: pd.DataFrame) -> None:
    missing = [column for column in REQUIRED_COLUMNS if column not in dataframe.columns]
    if missing:
        raise ValueError(
            "Missing required columns: "
            + ", ".join(missing)
            + ". Required: "
            + ", ".join(REQUIRED_COLUMNS)
        )


def compute_rate_percent(series: pd.Series) -> float:
    return float(series.astype(float).mean() * 100.0)


def compute_kpis(dataframe: pd.DataFrame, baseline_perplexity: float, finetuned_perplexity: float) -> dict:
    if baseline_perplexity <= 0 or finetuned_perplexity <= 0:
        raise ValueError("Perplexity values must be positive.")

    perplexity_improvement = ((baseline_perplexity - finetuned_perplexity) / baseline_perplexity) * 100.0

    return {
        "perplexity_improvement_percent": perplexity_improvement,
        "domain_relevance_rate_percent": compute_rate_percent(dataframe["is_domain_relevant"]),
        "hallucination_offtopic_rate_percent": compute_rate_percent(dataframe["is_hallucinated_or_offtopic"]),
        "repetition_rate_percent": compute_rate_percent(dataframe["is_repetitive"]),
        "draft_usability_rate_percent": compute_rate_percent(dataframe["is_draft_usable"]),
    }


def print_kpi_report(kpi_values: dict, sample_count: int) -> None:
    print("\nNon-Instruction Model Evaluation Report")
    print("-" * 50)
    print(f"Sample size: {sample_count}")

    for kpi_name, explanation in KPI_DEFINITIONS.items():
        print(f"\n{kpi_name}:")
        print(f"  Explanation: {explanation}")
        print(f"  Value: {kpi_values[kpi_name]:.2f}%")

    print("\nSuggested Starter Targets")
    print("  perplexity_improvement_percent >= 10")
    print("  domain_relevance_rate_percent >= 85")
    print("  hallucination_offtopic_rate_percent <= 10")
    print("  repetition_rate_percent <= 10")
    print("  draft_usability_rate_percent >= 70")

## Step 5: Run with Demo Data (Recommended First Run)

In [12]:
baseline_perplexity = 24.5
finetuned_perplexity = 19.8

dataframe = build_demo_dataframe()
validate_required_columns(dataframe)
kpi_values = compute_kpis(dataframe, baseline_perplexity, finetuned_perplexity)
print_kpi_report(kpi_values, sample_count=len(dataframe))


Non-Instruction Model Evaluation Report
--------------------------------------------------
Sample size: 4

perplexity_improvement_percent:
  Explanation: How much perplexity improved after fine-tuning compared with baseline. Higher is better.
  Value: 19.18%

domain_relevance_rate_percent:
  Explanation: Percent of outputs that stay in finance/credit-card context. Higher is better.
  Value: 75.00%

hallucination_offtopic_rate_percent:
  Explanation: Percent of outputs containing fabricated or off-topic content. Lower is better.
  Value: 25.00%

repetition_rate_percent:
  Explanation: Percent of outputs with looping/repeated phrases. Lower is better.
  Value: 25.00%

draft_usability_rate_percent:
  Explanation: Percent of outputs usable as a first draft with minimal edits. Higher is better.
  Value: 50.00%

Suggested Starter Targets
  perplexity_improvement_percent >= 10
  domain_relevance_rate_percent >= 85
  hallucination_offtopic_rate_percent <= 10
  repetition_rate_percent <= 10
  

## Step 6: Run with Your CSV
Set your CSV path and perplexity values, then run this cell.

In [13]:
USE_USER_CSV = False
USER_CSV_PATH = "evaluation/your_labeled_outputs.csv"
BASELINE_PERPLEXITY = 24.5
FINETUNED_PERPLEXITY = 19.8

if USE_USER_CSV:
    user_df = load_user_csv(USER_CSV_PATH)
    validate_required_columns(user_df)
    user_kpis = compute_kpis(user_df, BASELINE_PERPLEXITY, FINETUNED_PERPLEXITY)
    print_kpi_report(user_kpis, sample_count=len(user_df))
else:
    print("Set USE_USER_CSV = True and update USER_CSV_PATH to run on your data.")

Set USE_USER_CSV = True and update USER_CSV_PATH to run on your data.


## Interpretation Guide
- Higher is better for: perplexity_improvement_percent, domain_relevance_rate_percent, draft_usability_rate_percent.
- Lower is better for: hallucination_offtopic_rate_percent, repetition_rate_percent.
- For business decisions, treat this non-instruction model as a baseline and compare against your instruction model on the same test set.